# Publication-Style Reproduction Notebook

This notebook is designed for deterministic, figure-oriented reproduction using shipped outputs, with optional script reruns.

## Reproduction Modes

- `RUN_REGENERATE = False` (default): use precomputed outputs in the repository.
- `RUN_REGENERATE = True`: rerun the generating scripts before figure checks.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CWD = Path.cwd().resolve()
ROOT = CWD if (CWD / 'flow_computations').exists() else CWD.parent
if not (ROOT / 'flow_computations').exists():
    # fallback: search upward a few levels
    for parent in CWD.parents:
        if (parent / 'flow_computations').exists():
            ROOT = parent
            break
FLOW = ROOT / 'flow_computations'
DIPS = ROOT / 'dip_observations'
RUN_REGENERATE = False

if not FLOW.exists() or not DIPS.exists():
    raise RuntimeError('Run from repository root: flow_computations/ and dip_observations/ must exist.')

print('ROOT:', ROOT)
print('RUN_REGENERATE:', RUN_REGENERATE)


In [ ]:
def run(cmd, cwd, env=None):
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(cwd), check=True, env=env)

def show_png(path, title=None, width=12, height=5):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    img = plt.imread(path)
    plt.figure(figsize=(width, height))
    plt.imshow(img)
    plt.axis('off')
    plt.title(title or path.name)
    plt.show()

In [ ]:
# Figure/output manifest used in this notebook
manifest = [
    {
        'id': 'F-IDEALIZED-1',
        'desc': 'Pressure field: LargeSP_RetreatingTrench',
        'path': FLOW / 'plots' / 'pressure_fields' / 'LargeSP_RetreatingTrench.3e+20noslabflux.plotvisc4e+20.png',
        'dp': FLOW / 'text_files' / 'LargeSP_RetreatingTrench.3e+20noslabflux' / 'DP.txt',
    },
    {
        'id': 'F-IDEALIZED-2',
        'desc': 'Pressure field: LargeSP_RetreatingTrenchSlabGap',
        'path': FLOW / 'plots' / 'pressure_fields' / 'LargeSP_RetreatingTrenchSlabGap.3e+20noslabflux.plotvisc4e+20.png',
        'dp': FLOW / 'text_files' / 'LargeSP_RetreatingTrenchSlabGap.3e+20noslabflux' / 'DP.txt',
    },
    {
        'id': 'F-GLOBAL-P',
        'desc': 'Pressure field: Slab2.0Final_NoJapTail_nnr_FS',
        'path': FLOW / 'plots' / 'pressure_fields' / 'Slab2.0Final_NoJapTail_nnr_FS.3e+20_VcSlabFlux_width500000.0_alpha0.0.NoTailFlux.plotvisc4e+20.png',
        'dp': FLOW / 'text_files' / 'Slab2.0Final_NoJapTail_nnr_FS.3e+20_VcSlabFlux_width500000.0_alpha0.0.NoTailFlux' / 'DP.txt',
    },
    {
        'id': 'F-GLOBAL-DIP',
        'desc': 'Dip comparison: Slab2.0Final_NoJapTail_nnr_FS',
        'path': FLOW / 'plots' / 'dip_comparisons' / 'Slab2.0Final_NoJapTail_nnr_FS.3e+20_VcSlabFlux_width500000.0_alpha0.0.NoTailFlux.fact1.327.png',
        'dp': FLOW / 'text_files' / 'Slab2.0Final_NoJapTail_nnr_FS.3e+20_VcSlabFlux_width500000.0_alpha0.0.NoTailFlux' / 'DP.txt',
    },
    {
        'id': 'F-DIPS-COMPARE',
        'desc': 'Observed dip catalogue comparison',
        'path': DIPS / 'plots' / 'compare_all_dips.png',
        'dp': DIPS / 'dip_catalogues' / 'Slab2_const-depth' / 'AllDips.txt',
    },
]

manifest_df = pd.DataFrame([{
    'id': m['id'],
    'description': m['desc'],
    'figure_path': str(m['path'].relative_to(ROOT)),
    'data_path': str(m['dp'].relative_to(ROOT)),
    'figure_exists': m['path'].exists(),
    'data_exists': m['dp'].exists(),
} for m in manifest])

manifest_df

## Optional regeneration (script reruns)

Set `RUN_REGENERATE = True` to run this cell. It regenerates one global pressure case, ages, dip comparison, and dip-observation comparison figure.

In [ ]:
if RUN_REGENERATE:
    env = os.environ.copy()
    env['DIPS_OBS_TXT'] = str(DIPS / 'dip_catalogues' / 'Slab2_const-depth' / 'AllDips.txt')

    run([sys.executable, 'global_pressure_withPressurePlot.py',
         'Slab2.0Final_NoJapTail_nnr_FS', '3.0e20', '2', '500000', '0', '1', 'Subgrd.inp', '4.0e20'],
        cwd=FLOW, env=env)

    run([sys.executable, 'get_SPages.py', 'Slab2.0Final_NoJapTail_nnr_FS'], cwd=FLOW, env=env)

    run([sys.executable, 'plot_DipComparison_varyDPfactor.py',
         'Slab2.0Final_NoJapTail_nnr_FS', '3.0e20', '2', '500000', '0', '1', 'Subgrd.inp', '2', '12', '4', '5'],
        cwd=FLOW, env=env)

    run([sys.executable, 'compare_all_dips.py'], cwd=DIPS, env=env)
else:
    print('Skipping regeneration; RUN_REGENERATE is False')

## DP summary table for reproducibility checks

In [ ]:
rows = []
for m in manifest:
    data_path = m['dp']
    if not data_path.exists():
        rows.append({'id': m['id'], 'n': np.nan, 'mean': np.nan, 'median': np.nan, 'min': np.nan, 'max': np.nan})
        continue

    arr = np.loadtxt(data_path)
    vec = arr[:,4] if arr.ndim == 2 and arr.shape[1] >= 5 else arr[:,2] if arr.ndim == 2 and arr.shape[1] >= 3 else arr
    rows.append({
        'id': m['id'],
        'n': int(vec.shape[0]),
        'mean': float(np.mean(vec)),
        'median': float(np.median(vec)),
        'min': float(np.min(vec)),
        'max': float(np.max(vec)),
    })

pd.DataFrame(rows).set_index('id').round(4)

## Figure panels (publication-style)

In [ ]:
for m in manifest:
    print(f"\n[{m['id']}] {m['desc']}")
    show_png(m['path'], title=f"{m['id']} | {m['desc']}")

In [ ]:
# Optional: write manifest to CSV for archiving
out_csv = ROOT / 'notebooks' / 'publication_manifest.csv'
manifest_df.to_csv(out_csv, index=False)
print('Wrote', out_csv.relative_to(ROOT))